# Structured Outputs

LLMs regurgitate out text and that is great for so many applications. But in order to build strong, robust systems and applications, we need to make sense of the chaos sometimes by receiving a pre-determined structured output everytime an LLM is called.

## As always, libraries first!

In [11]:
import pandas as pd # for data manipulation and analysis
from pydantic import BaseModel # for data validation and settings management

from config import Config # for configuration management

import os # for environment variable management
from openai import OpenAI # for interacting with OpenAI's API
from dotenv import load_dotenv # for managing environment variables
from IPython.display import display, Markdown # for displaying output
from anthropic import Anthropic # for interacting with Anthropic's API
import google.genai as genai # for interacting with Google's Generative AI API
from google.genai import types # for interacting with Google's Generative AI API
# import google.generativeai as genai # for interacting with Google's Generative AI API

In [2]:
# check if API keys are set
# if not Config.OPENAI_API_KEY:
#     raise ValueError("Missing OpenAI API key")
# if not Config.GEMINI_API_KEY:
#     raise ValueError("Missing Gemini API key")
# if not Config.ANTHROPIC_API_KEY:
#     raise ValueError("Missing Anthropic API key")

# Fail-fast validation before initialization [cite: 225]
if not all([Config.OPENAI_API_KEY, Config.GEMINI_API_KEY, Config.ANTHROPIC_API_KEY]):
    raise ValueError("Missing one or more required API keys in .env")

google_client = genai.Client(api_key=Config.GEMINI_API_KEY)
anthropic_client = Anthropic(api_key=Config.ANTHROPIC_API_KEY)
openai_client = OpenAI(api_key=Config.OPENAI_API_KEY)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## The Workflow

```mermaid
graph LR
    A[Generate Ticket] --> B[Respond to Ticket]
    B --> C[Evaluate Response]
    C --> B
    C --> D[Final Output]
```

## Introducing the Pydantic library

In [4]:
# 'BaseModel' is a class from the Pydantic library that provides data validation.
# The following classes helps define the structure and data types for our models

class CustomerTicket(BaseModel):
    ticket: str
    priority: str
    assigned_to: str

class TicketResponse(BaseModel):
    response: str
    resolution_time: str

class TicketEvaluation(BaseModel):
    passed: bool
    feedback: str

## Calling OpenAI to generate support tickets

In [5]:
# messages list
user_message = "I want you to generate a customer support ticket for a 3rd party tech re-seller. "
user_message += "The ticket should be a single sentence describing a common issue a customer might face with their product or service. "
user_message += "Please ensure the ticket is varied and covers different types of problems."

messages = [{"role": "user", "content": user_message}]

In [ ]:
# normal response
response = openai_client.chat.completions.create(
    model= Config.DEFAULT_OPENAI_MODEL,
    messages=messages
)

normal_response = response.choices[0].message.content
display(Markdown(f"### Normal Response:\n{normal_response}"))

### Normal Response:
The customer is unable to activate their purchased software license due to repeated error messages during the installation process.

In [7]:
# structured response, will generate a "random" ticket that fits the CustomerTicket schema
structured_response = openai_client.chat.completions.parse(
    model = Config.DEFAULT_OPENAI_MODEL,
    messages=messages,
    # new, specify the response format
    response_format = CustomerTicket
)
# new, access message.parsed instead of message.content
structured_response = structured_response.choices[0].message.parsed
display(Markdown(f"### Structured Response:\n{structured_response}"))

### Structured Response:
ticket='The customer reports that their recently purchased software is not activating despite multiple attempts.' priority='High' assigned_to='Technical Support Team'

In [8]:
structured_response.ticket

'The customer reports that their recently purchased software is not activating despite multiple attempts.'

In [9]:
structured_response.priority

'High'

In [14]:
df_ticket = pd.DataFrame(
    [{"Ticket": structured_response.ticket, "Priority": structured_response.priority, "Assigned To": structured_response.assigned_to}],
    columns=["Ticket", "Priority", "Assigned To"]
)
display(df_ticket)

,Ticket,Priority,Assigned To
0,The customer reports that their recently purch...,High,Technical Support Team


## Responding to the ticket

In [15]:
# messages list
message = "You are to propose a resolution for the following customer support ticket. \n\n"
message += f"Ticket: {structured_response.ticket}\n"
message += f"Priority: {structured_response.priority}\n\n"

messages = [{"role": "user", "content": message}]

In [16]:
# structured response
ticket_response = openai_client.chat.completions.parse(
    model=Config.DEFAULT_OPENAI_MODEL,
    messages=messages,
    response_format=TicketResponse
)

ticket_response = ticket_response.choices[0].message.parsed
display(Markdown(f"### Response:\n{ticket_response.response}"))
display(Markdown(f"### Resolution Time:\n{ticket_response.resolution_time}"))

### Response:
Dear Customer,

Thank you for reaching out regarding the activation issue with your recently purchased software. We understand the urgency of this matter and apologize for any inconvenience caused.

To assist you effectively, please ensure that your device meets the software's system requirements and that you have a stable internet connection. We recommend the following steps:

1. Restart your device.
2. Disable any antivirus or firewall temporarily that might be blocking activation.
3. Run the software as an administrator (right-click the icon and select 'Run as administrator').
4. Enter the activation key again carefully, ensuring there are no typos.

If the issue persists, please provide us with the following information:
- The exact error message displayed during activation.
- Your operating system and version.
- The activation key or license ID.

Once we receive these details, our technical support team will investigate further and guide you towards a resolution promptly.

Thank you for your patience and understanding.

Best regards,
Customer Support Team

### Resolution Time:
Within 24 hours

## Lets evaluate our response

In [17]:
# messages list
message = "You are to evaluate the proposed resolution for the following customer support ticket. "
message += "You will determine if the proposed resolution is appropriate for the ticket and priority level. "
message += "tickets\n\n"
message += f"Ticket: {structured_response.ticket}\n"
message += f"Priority: {structured_response.priority}\n\n"
message += f"Proposed Resolution: {ticket_response.response}\n"
message += f"Proposed Resolution {ticket_response.resolution_time}\n\n"

messages = [{"role": "user", "content": message}]

In [18]:
messages

[{'role': 'user',
  'content': "You are to evaluate the proposed resolution for the following customer support ticket. You will determine if the proposed resolution is appropriate for the ticket and priority level. tickets\n\nTicket: The customer reports that their recently purchased software is not activating despite multiple attempts.\nPriority: High\n\nProposed Resolution: Dear Customer,\n\nThank you for reaching out regarding the activation issue with your recently purchased software. We understand the urgency of this matter and apologize for any inconvenience caused.\n\nTo assist you effectively, please ensure that your device meets the software's system requirements and that you have a stable internet connection. We recommend the following steps:\n\n1. Restart your device.\n2. Disable any antivirus or firewall temporarily that might be blocking activation.\n3. Run the software as an administrator (right-click the icon and select 'Run as administrator').\n4. Enter the activation k

In [19]:
# evaluate response
evaluator_response = openai_client.chat.completions.parse(
    model=Config.DEFAULT_OPENAI_MODEL,
    messages=messages,
    response_format=TicketEvaluation
)

evaluator_response = evaluator_response.choices[0].message.parsed
display(Markdown(f"### Passed:\n{evaluator_response.passed}"))
display(Markdown(f"### Feedback:\n{evaluator_response.feedback}"))

### Passed:
True

### Feedback:
The proposed resolution provides a comprehensive set of troubleshooting steps suitable for high-priority activation issues. It appropriately acknowledges the urgency, offers clear instructions, and requests necessary additional information. The plan to follow up within 24 hours aligns with high-priority customer support standards. Overall, the resolution is appropriate and well-structured.

<div style="border-radius:16px;background:#1e2a1e;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
  <b style="color:#a3be8c;font-size:1.25em">Your Challenge:</b>
  <ul style="margin:.6em 0 0;padding-left:1.2em;line-height:1.6">
    <li>Hey everyone! Ready to flex those agentic muscles? 🎉 Build a workflow just like the ticket system above, but for <b>product reviews</b>!</li>
    <li>Your workflow should:
      <ul>
        <li>Generate a product review (think: electronics, books, or your favorite kitchen gadget)</li>
        <li>Respond to the review (company reply, moderation, or a witty bot response)</li>
        <li>Evaluate the response (is it helpful, polite, and on point?)</li>
      </ul>
    </li>
    <li>Use structured outputs and Pydantic models for each step, just like we did above.</li>
    <li>Include an evaluator step to assess the quality of the response.</li>
    <li>Here’s a suggested workflow to get your creative gears turning:</li>
  </ul>
  <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#a3be8c;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💪</div>
</div>

### Suggested Workflow

```mermaid
graph LR
    A[Generate Review] --> B[Respond to Review]
    B --> C[Evaluate Response]
    C --> B
    C --> D[Final Output]
```

Try to use structured outputs and Pydantic models for each step, just like in the notebook above. Include an evaluator step to assess the quality of the response.